# Silver: clean, type, deduplicate, quarantine, then gate

Rebuilds the `silver_*` tables from bronze. Bad rows go to `silver_quarantine` with reasons, trailers to `silver_control_totals`, one audit row per entity to `silver_load_audit`. Then runs the silver data quality checks and fails the run if an ERROR check fails.

Requires the `env_portfolio` Environment (with the `portfolio_migration` wheel) and the `lh_portfolio` lakehouse attached as default. Logic and tests live in the GitHub repo; do not edit logic here.

In [ ]:
# Set by the data pipeline to @pipeline().RunId. Empty means a manual run.
batch_id = ""

In [ ]:
import json
import time
import uuid
from dataclasses import asdict

import portfolio_migration
from portfolio_migration.lakehouse.io import FabricLake

batch_id = batch_id or f"manual-{uuid.uuid4()}"
print(f"portfolio_migration {portfolio_migration.__version__}, batch {batch_id}")
lake = FabricLake(spark)
started = time.perf_counter()

In [ ]:
from portfolio_migration.lakehouse import dq_checks, quality, silver

LAYER = "silver"
output = [asdict(r) for r in silver.run(lake, batch_id)]
display(spark.createDataFrame(output))

# Gate: raises DataQualityError if any ERROR check fails, which fails the pipeline
# and stops gold from running. Results are written to dq_results either way.
checks = quality.run(lake, dq_checks.SILVER_CHECKS, batch_id, gate=True)
display(spark.createDataFrame([asdict(c) for c in checks]))
print(f"{len(checks)} silver checks, "
      f"{len(quality.failures(checks, quality.WARN))} warnings")

In [ ]:
seconds = round(time.perf_counter() - started, 1)
print(f"finished in {seconds} s")
notebookutils.notebook.exit(json.dumps({"layer": LAYER, "batch_id": batch_id, "seconds": seconds,
                                        "output": output}))